# Credit Risk Scorecard — Bước 4: Validation, Calibration & Model Card
**Input:** `model_lr.pkl` · `cs_scores.csv` · `cs_woe_transformed.csv` · `woe_bins.pkl`  
**Output:** `model_card.md` · `psi_report.csv` · `validation_report.png`

---
### Cấu trúc notebook
1. Reload artifacts từ Bước 3
2. Calibration check — Reliability Diagram
3. Threshold optimization — Precision / Recall trade-off
4. Population Stability Index (PSI)
5. Gains & Lift Table
6. Backtesting — Basel Traffic Light
7. Top Reason Codes (giải thích từng khách hàng)
8. Tổng hợp Validation Report
9. Xuất Model Card (Markdown)


## Tại sao cần Validation riêng sau khi có AUC?

AUC chỉ đo khả năng **xếp hạng** (ranking) — không đảm bảo xác suất được calibrate đúng.  
Ví dụ: model có AUC = 0.80 nhưng predict P(bad) = 0.03 cho khách hàng thực ra có bad rate 8%  
→ **Ngân hàng định giá lãi suất sai** → thua lỗ.

Validation đầy đủ cần kiểm tra thêm:

| Kiểm tra | Câu hỏi | Công cụ |
|---|---|---|
| Calibration | Xác suất predict có đúng không? | Reliability diagram, Brier score |
| Stability | Model có drift theo thời gian không? | PSI |
| Business value | Có thực sự giúp capture bad hơn random? | Gains/Lift table |
| Regulatory | Có đạt ngưỡng Basel không? | Traffic light test |
| Explainability | Tại sao khách hàng bị từ chối? | Reason codes |


## 1. Reload artifacts từ Bước 3

In [ ]:
import warnings, pickle, json
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.calibration   import calibration_curve, CalibratedClassifierCV
from sklearn.metrics       import (roc_auc_score, roc_curve, brier_score_loss,
                                   precision_recall_curve, average_precision_score,
                                   confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split
import scorecardpy as sc

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
plt.rcParams.update({
    "figure.dpi": 115,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "font.family": "sans-serif",
})
PALETTE = {"good": "#2196F3", "bad": "#E53935", "neutral": "#78909C"}

# ── Load artifacts ────────────────────────────────────────────────────────────
with open("model_lr.pkl",     "rb") as f: lr_model   = pickle.load(f)
with open("woe_bins.pkl",     "rb") as f: bins_final = pickle.load(f)
with open("selected_vars.pkl","rb") as f: SELECTED_VARS = pickle.load(f)
with open("model_metadata.json") as f:   metadata   = json.load(f)

df_woe    = pd.read_csv("cs_woe_transformed.csv")
score_all = pd.read_csv("cs_scores.csv")

# Feature columns thực tế model đang dùng
WOE_COLS_FINAL = metadata["features"]

X = df_woe[WOE_COLS_FINAL]
y = df_woe["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

y_prob_test  = lr_model.predict_proba(X_test)[:, 1]
y_prob_train = lr_model.predict_proba(X_train)[:, 1]

auc_test = roc_auc_score(y_test, y_prob_test)
print(f"✓ Artifacts loaded")
print(f"  Features : {len(WOE_COLS_FINAL)}")
print(f"  AUC test : {auc_test:.4f}")
print(f"  Metadata : {metadata}")


## 2. Calibration Check — Reliability Diagram

Một model được calibrate tốt: khi nó predict P(bad) = 10%, thì trong nhóm đó  
thực sự ~10% khách hàng bị default. Đường calibration nên **gần đường diagonal**.

**Brier Score** = mean squared error của probability predictions:
- Brier = 0 → perfect
- Brier = 0.25 → random (với balanced dataset)
- Thực tế credit model: Brier ~0.04–0.06 là tốt (do base rate thấp)


In [ ]:
# ── Calibration trước và sau Platt Scaling ───────────────────────────────────
calibrated_lr = CalibratedClassifierCV(lr_model, method="sigmoid", cv="prefit")
calibrated_lr.fit(X_train, y_train)
y_prob_cal = calibrated_lr.predict_proba(X_test)[:, 1]

# Reliability curves
frac_pos_raw, mean_pred_raw = calibration_curve(y_test, y_prob_test, n_bins=10)
frac_pos_cal, mean_pred_cal = calibration_curve(y_test, y_prob_cal,  n_bins=10)

# Brier scores
brier_raw = brier_score_loss(y_test, y_prob_test)
brier_cal = brier_score_loss(y_test, y_prob_cal)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Reliability diagram ───────────────────────────────────────────────────────
axes[0].plot([0,1],[0,1], "k--", lw=1, label="Perfect calibration")
axes[0].plot(mean_pred_raw, frac_pos_raw, "o-",
             color=PALETTE["bad"],  lw=2, ms=7,
             label=f"Before calibration (Brier={brier_raw:.4f})")
axes[0].plot(mean_pred_cal, frac_pos_cal, "s-",
             color=PALETTE["good"], lw=2, ms=7,
             label=f"After Platt scaling (Brier={brier_cal:.4f})")
axes[0].set_xlabel("Mean predicted probability")
axes[0].set_ylabel("Fraction of positives (actual bad rate)")
axes[0].set_title("Reliability Diagram (Calibration)")
axes[0].legend(fontsize=9, frameon=False)
axes[0].set_xlim(0, 0.5); axes[0].set_ylim(0, 0.5)

# ── Probability histogram ─────────────────────────────────────────────────────
axes[1].hist(y_prob_test[y_test==0], bins=50, alpha=0.6,
             color=PALETTE["good"], label="Good", density=True, edgecolor="none")
axes[1].hist(y_prob_test[y_test==1], bins=50, alpha=0.6,
             color=PALETTE["bad"],  label="Bad",  density=True, edgecolor="none")
axes[1].set_xlabel("Predicted P(bad)")
axes[1].set_ylabel("Density")
axes[1].set_title("Predicted Probability Distribution")
axes[1].legend(fontsize=9, frameon=False)

plt.suptitle("Calibration Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("14_calibration.png", bbox_inches="tight")
plt.show()

improvement = (brier_raw - brier_cal) / brier_raw * 100
print(f"Brier score improvement after calibration: {improvement:.1f}%")
if improvement > 0:
    print("✓ Platt scaling cải thiện calibration — dùng calibrated_lr cho production")
else:
    print("→ Calibration không cải thiện nhiều — dùng lr_model gốc cũng được")


## 3. Threshold Optimization — Precision / Recall

Với imbalanced dataset (bad rate 6.7%), F1-score và Precision-Recall  
quan trọng hơn Accuracy. Ta tìm threshold tối ưu theo **F1-score**.

- **Precision** = trong số bị reject, bao nhiêu % thực sự là Bad?
- **Recall** = trong số Bad thực, bao nhiêu % bị model bắt được?
- **F1** = harmonic mean của Precision và Recall


In [ ]:
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_prob_test)
ap_score = average_precision_score(y_test, y_prob_test)

# F1 tại mỗi threshold
f1_scores = np.where(
    (precision[:-1] + recall[:-1]) > 0,
    2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1]),
    0
)
best_idx = np.argmax(f1_scores)
best_thr = pr_thresholds[best_idx]
best_f1  = f1_scores[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── PR Curve ──────────────────────────────────────────────────────────────────
axes[0].plot(recall, precision, color=PALETTE["good"], lw=2,
             label=f"PR curve (AP={ap_score:.3f})")
axes[0].scatter(recall[best_idx], precision[best_idx],
                color=PALETTE["bad"], s=100, zorder=5,
                label=f"Best F1={best_f1:.3f} @ thr={best_thr:.3f}")
axes[0].axhline(y_test.mean(), color="gray", linestyle="--", lw=1,
                label=f"Baseline (bad rate={y_test.mean()*100:.1f}%)")
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title(f"Precision-Recall Curve (AP = {ap_score:.3f})")
axes[0].legend(fontsize=9, frameon=False)

# ── F1 vs Threshold ───────────────────────────────────────────────────────────
axes[1].plot(pr_thresholds, f1_scores, color="#7B1FA2", lw=2, label="F1 score")
axes[1].plot(pr_thresholds, precision[:-1], color=PALETTE["good"],
             lw=1.5, linestyle="--", alpha=0.7, label="Precision")
axes[1].plot(pr_thresholds, recall[:-1],    color=PALETTE["bad"],
             lw=1.5, linestyle="--", alpha=0.7, label="Recall")
axes[1].axvline(best_thr, color="orange", linestyle=":", lw=2,
                label=f"Optimal threshold = {best_thr:.3f}")
axes[1].set_xlabel("Threshold")
axes[1].set_ylabel("Score")
axes[1].set_title("F1, Precision, Recall vs Threshold")
axes[1].legend(fontsize=9, frameon=False)
axes[1].set_xlim(0, 0.5)

plt.tight_layout()
plt.savefig("15_precision_recall.png", bbox_inches="tight")
plt.show()

# Confusion matrix tại optimal threshold
y_pred_opt = (y_prob_test >= best_thr).astype(int)
cm = confusion_matrix(y_test, y_pred_opt)
print(f"\nOptimal threshold: {best_thr:.4f}  |  F1: {best_f1:.4f}")
print(f"\nConfusion Matrix (threshold = {best_thr:.3f}):")
print(f"  TN={cm[0,0]:,}  FP={cm[0,1]:,}")
print(f"  FN={cm[1,0]:,}  TP={cm[1,1]:,}")
print()
print(classification_report(y_test, y_pred_opt, target_names=["Good","Bad"]))


## 4. Population Stability Index (PSI)

PSI đo lường **drift** của phân phối score giữa thời điểm train và deploy (hoặc giữa các kỳ).  
Trong thực tế: so sánh score distribution của train set (baseline) vs test set (recent).

$$PSI = \sum_{i} (\%Actual_i - \%Expected_i) \times \ln\left(\frac{\%Actual_i}{\%Expected_i}\right)$$

| PSI | Kết luận |
|---|---|
| < 0.10 | Ổn định — không cần action |
| 0.10 – 0.25 | Theo dõi thêm |
| > 0.25 | Drift nghiêm trọng — cần retrain |


In [ ]:
def compute_psi(expected: np.ndarray, actual: np.ndarray,
                n_bins: int = 10) -> tuple[float, pd.DataFrame]:
    """
    Tính PSI giữa expected (train) và actual (test) score distributions.
    
    Returns
    -------
    psi_total : float
    psi_table : pd.DataFrame — chi tiết từng bin
    """
    # Tạo bins dựa trên expected distribution
    breakpoints = np.percentile(expected, np.linspace(0, 100, n_bins + 1))
    breakpoints  = np.unique(breakpoints)  # loại duplicate

    exp_counts = np.histogram(expected, bins=breakpoints)[0]
    act_counts = np.histogram(actual,   bins=breakpoints)[0]

    exp_pct = exp_counts / len(expected)
    act_pct = act_counts / len(actual)

    # Tránh log(0)
    eps = 1e-6
    exp_pct = np.where(exp_pct == 0, eps, exp_pct)
    act_pct = np.where(act_pct == 0, eps, act_pct)

    psi_bins = (act_pct - exp_pct) * np.log(act_pct / exp_pct)
    psi_total = psi_bins.sum()

    labels = [f"{breakpoints[i]:.0f}–{breakpoints[i+1]:.0f}"
              for i in range(len(breakpoints)-1)]

    psi_table = pd.DataFrame({
        "Score Band":    labels,
        "Expected %":   (exp_pct * 100).round(2),
        "Actual %":     (act_pct * 100).round(2),
        "PSI":           psi_bins.round(5),
    })
    return psi_total, psi_table

# Score từ train/test (dùng index từ split)
train_idx, test_idx = train_test_split(
    score_all.index, test_size=0.3, random_state=42,
    stratify=score_all["target"]
)
score_train_vals = score_all.loc[train_idx, "score"].values
score_test_vals  = score_all.loc[test_idx,  "score"].values

psi_total, psi_table = compute_psi(score_train_vals, score_test_vals, n_bins=10)

print(f"PSI (train vs test): {psi_total:.4f}")
if psi_total < 0.10:
    status = "✓ Ổn định — model stable"
elif psi_total < 0.25:
    status = "⚠️  Theo dõi thêm — minor shift"
else:
    status = "✗ Drift nghiêm trọng — cần retrain"
print(f"Status: {status}")
print()
print(psi_table.to_string(index=False))

psi_table.to_csv("psi_report.csv", index=False)
print("\n✓ Saved: psi_report.csv")

# ── Visualize PSI ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(psi_table))
w = 0.35
axes[0].bar(x - w/2, psi_table["Expected %"], w, label="Train (Expected)",
            color=PALETTE["good"]+"88", edgecolor="white")
axes[0].bar(x + w/2, psi_table["Actual %"],   w, label="Test (Actual)",
            color=PALETTE["bad"]+"88",  edgecolor="white")
axes[0].set_xticks(x)
axes[0].set_xticklabels(psi_table["Score Band"], rotation=35, ha="right", fontsize=8)
axes[0].set_ylabel("Population %")
axes[0].set_title(f"Score Distribution: Train vs Test  |  PSI = {psi_total:.4f}")
axes[0].legend(fontsize=9)

axes[1].bar(x, psi_table["PSI"], color=[
    "#E53935" if v > 0.025 else "#FFB300" if v > 0.01 else "#66BB6A"
    for v in psi_table["PSI"]
], width=0.6, edgecolor="white")
axes[1].axhline(0.025, color="orange", linestyle="--", lw=1.5, label="0.025 threshold")
axes[1].set_xticks(x)
axes[1].set_xticklabels(psi_table["Score Band"], rotation=35, ha="right", fontsize=8)
axes[1].set_ylabel("PSI per bin")
axes[1].set_title("PSI Contribution by Score Band")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig("16_psi_analysis.png", bbox_inches="tight")
plt.show()


## 5. Gains & Lift Table

**Gains table** cho biết: nếu ta contact/review top X% khách hàng rủi ro nhất,  
ta capture được bao nhiêu % total Bad?

**Lift** = (% Bad captured) / (% Population reviewed) — Lift > 1 có nghĩa là tốt hơn random.


In [ ]:
def gains_lift_table(y_true: np.ndarray, y_prob: np.ndarray,
                     n_deciles: int = 10) -> pd.DataFrame:
    """
    Xây dựng Gains & Lift table theo decile.
    Sắp xếp từ rủi ro cao nhất (decile 1) đến thấp nhất (decile 10).
    """
    df_gl = pd.DataFrame({"target": y_true, "prob": y_prob})
    df_gl = df_gl.sort_values("prob", ascending=False).reset_index(drop=True)
    df_gl["decile"] = pd.qcut(df_gl.index, n_deciles,
                               labels=range(1, n_deciles+1))

    total_bad  = y_true.sum()
    total_good = (1 - y_true).sum()
    total_n    = len(y_true)

    rows = []
    cum_bad = cum_good = cum_n = 0
    for d in range(1, n_deciles+1):
        grp = df_gl[df_gl["decile"] == d]
        n       = len(grp)
        n_bad   = grp["target"].sum()
        n_good  = n - n_bad
        cum_n   += n
        cum_bad += n_bad
        cum_good+= n_good
        rows.append({
            "Decile":          d,
            "N":               n,
            "N Bad":           int(n_bad),
            "Bad Rate %":      round(n_bad/n*100, 2) if n > 0 else 0,
            "Cumulative N %":  round(cum_n/total_n*100, 1),
            "Cumulative Bad %":round(cum_bad/total_bad*100, 1),
            "Lift":            round((cum_bad/total_bad) / (cum_n/total_n), 3),
        })
    return pd.DataFrame(rows)

gl_table = gains_lift_table(y_test.values, y_prob_test)
print("Gains & Lift Table (Test Set):")
print(gl_table.to_string(index=False))

# ── Visualize ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gains chart
axes[0].plot([0] + list(gl_table["Cumulative N %"]),
             [0] + list(gl_table["Cumulative Bad %"]),
             color=PALETTE["good"], lw=2.5, marker="o", ms=6,
             label="Model")
axes[0].plot([0, 100], [0, 100], color="gray", linestyle="--", lw=1.5, label="Random")
axes[0].fill_between(
    [0] + list(gl_table["Cumulative N %"]),
    [0] + list(gl_table["Cumulative Bad %"]),
    [0] + list(gl_table["Cumulative N %"]),
    alpha=0.07, color=PALETTE["good"]
)
# Annotate top 3 deciles
for _, row in gl_table.head(3).iterrows():
    axes[0].annotate(
        f"{row['Cumulative Bad %']:.0f}%",
        xy=(row["Cumulative N %"], row["Cumulative Bad %"]),
        xytext=(row["Cumulative N %"]+3, row["Cumulative Bad %"]-5),
        fontsize=8.5, color=PALETTE["good"],
        arrowprops=dict(arrowstyle="-", color=PALETTE["good"], lw=0.8),
    )
axes[0].set_xlabel("% Population (sorted by score desc)")
axes[0].set_ylabel("% Bad captured")
axes[0].set_title("Cumulative Gains Chart")
axes[0].legend(fontsize=9, frameon=False)

# Lift chart
axes[1].bar(gl_table["Decile"], gl_table["Lift"],
            color=[PALETTE["bad"] if l > 1.5 else
                   PALETTE["neutral"] if l > 1 else
                   PALETTE["good"] + "88"
                   for l in gl_table["Lift"]],
            width=0.6, edgecolor="white")
axes[1].axhline(1.0, color="black", linestyle="--", lw=1.5, label="Baseline (Lift=1)")
for i, row in gl_table.iterrows():
    axes[1].text(row["Decile"], row["Lift"] + 0.03,
                 f"{row['Lift']:.2f}", ha="center", fontsize=8.5, fontweight="bold")
axes[1].set_xlabel("Decile (1 = highest risk)")
axes[1].set_ylabel("Lift")
axes[1].set_title("Lift by Decile")
axes[1].legend(fontsize=9)

plt.suptitle("Gains & Lift Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("17_gains_lift.png", bbox_inches="tight")
plt.show()

print(f"\nKey insight: Top 20% rủi ro nhất capture được "
      f"{gl_table.loc[gl_table['Decile']==2, 'Cumulative Bad %'].values[0]:.0f}% "
      f"tổng số Bad")


## 6. Backtesting — Basel Traffic Light Test

Basel II yêu cầu **backtesting** hàng năm: so sánh số lượng exceptions  
(predicted good nhưng thực tế default) với ngưỡng cho phép.

| Vùng | Exceptions / 250 obs | Ý nghĩa |
|---|---|---|
| Xanh | 0–4 | Model tốt |
| Vàng | 5–9 | Cần theo dõi |
| Đỏ | ≥ 10 | Model có vấn đề — điều tra ngay |

*250 là số ngày giao dịch trong 1 năm (convention của Basel cho market risk,  
áp dụng tương tự cho credit backtesting theo tỷ lệ)*


In [8]:
def basel_traffic_light(y_true: np.ndarray, y_prob: np.ndarray,
                        threshold: float, n_obs: int = 250) -> dict:
    """
    Basel Traffic Light test cho credit model.
    Exception = model predict Good (prob < threshold) nhưng thực tế Bad.
    
    Parameters
    ----------
    threshold : float — ngưỡng phân loại Good/Bad
    n_obs     : int   — số observations trong backtest window (default 250)
    """
    from scipy.stats import binom

    # Sample n_obs từ test set
    np.random.seed(42)
    idx = np.random.choice(len(y_true), min(n_obs, len(y_true)), replace=False)
    y_s = y_true[idx]
    p_s = y_prob[idx]

    predicted_good = p_s < threshold
    exceptions = int((predicted_good & (y_s == 1)).sum())
    n_predicted_good = predicted_good.sum()

    # P-value: probability of getting >= exceptions under null (model correct)
    # Null: P(exception) = actual bad rate among predicted good
    actual_bad_rate = y_s[predicted_good].mean() if predicted_good.sum() > 0 else 0
    expected_bad_rate = y_true.mean()
    p_value = 1 - binom.cdf(exceptions - 1, n_predicted_good, expected_bad_rate)

    # Traffic light
    exception_rate = exceptions / n_obs
    if exceptions <= 4:
        zone, color = "GREEN",  "#4CAF50"
    elif exceptions <= 9:
        zone, color = "YELLOW", "#FFC107"
    else:
        zone, color = "RED",    "#F44336"

    return {
        "n_obs":            n_obs,
        "n_predicted_good": int(n_predicted_good),
        "exceptions":       exceptions,
        "exception_rate":   round(exception_rate * 100, 2),
        "zone":             zone,
        "color":            color,
        "p_value":          round(p_value, 4),
    }

# Test tại nhiều threshold
thresholds_test = [0.05, 0.07, 0.10, 0.15, 0.20]
print("Basel Traffic Light Results:")
print("─" * 65)
print(f"{'Threshold':>10} {'N Good':>8} {'Exceptions':>11} {'Rate':>7} {'Zone':>8} {'p-value':>9}")
print("─" * 65)

results_btl = {}
for thr in thresholds_test:
    r = basel_traffic_light(y_test.values, y_prob_test, threshold=thr)
    results_btl[thr] = r
    print(f"{thr:>10.2f} {r['n_predicted_good']:>8,} {r['exceptions']:>11} "
          f"{r['exception_rate']:>6.2f}% {r['zone']:>8}  {r['p_value']:>9.4f}")

print("─" * 65)
print()

# Recommended threshold
rec_thr = None
for thr, r in results_btl.items():
    if r["zone"] == "GREEN":
        rec_thr = thr
        break
if rec_thr:
    print(f"✓ Recommended threshold: {rec_thr} → Zone {results_btl[rec_thr]['zone']}")


NameError: name 'np' is not defined

## 7. Top Reason Codes — Giải thích từng quyết định

Reason codes là **bắt buộc theo luật** ở nhiều quốc gia (ECOA, GDPR Article 22).  
Ngân hàng phải gửi thư từ chối kèm 4–5 lý do cụ thể tại sao đơn vay bị reject.

Cách tính: với mỗi khách hàng, so sánh điểm của bin họ đang ở  
với điểm tối đa có thể đạt được của từng biến → phần thiếu nhiều nhất = reason code #1.


In [ ]:
import scorecardpy as sc

def get_reason_codes(customer_data: pd.Series, card: dict,
                     bins: dict, top_n: int = 5) -> pd.DataFrame:
    """
    Tính reason codes cho 1 khách hàng.
    
    Parameters
    ----------
    customer_data : pd.Series — 1 hàng từ raw data (KHÔNG phải WOE)
    card          : dict      — scorecard từ sc.scorecard()
    bins          : dict      — bins từ sc.woebin()
    top_n         : int       — số reason codes muốn trả về
    
    Returns
    -------
    DataFrame với cột: variable, customer_bin, customer_points,
                       max_points, points_gap, rank
    """
    rows = []
    for var, card_tbl in card.items():
        if var == "basepoints":
            continue
        raw_var = var.replace("_woe", "")
        if raw_var not in customer_data.index:
            continue

        val = customer_data[raw_var]
        bin_tbl = bins.get(raw_var, None)
        if bin_tbl is None:
            continue

        # Tìm bin của khách hàng
        customer_points = None
        customer_bin    = "unknown"
        for _, row in bin_tbl.iterrows():
            bin_str = str(row["bin"])
            # Parse bin range
            try:
                if "," in bin_str:
                    lo = float(bin_str.split(",")[0].replace("[","").replace("(","").replace("-inf","-999999"))
                    hi = float(bin_str.split(",")[1].replace(")","").replace("]","").replace("inf","999999"))
                    if lo <= val < hi:
                        customer_bin = bin_str
                        pts_match = card_tbl[card_tbl["bin"] == bin_str]
                        if not pts_match.empty:
                            customer_points = pts_match["points"].values[0]
                        break
            except Exception:
                continue

        max_points = card_tbl["points"].max()
        if customer_points is None:
            customer_points = card_tbl["points"].min()

        rows.append({
            "Variable":         raw_var,
            "Customer bin":     customer_bin,
            "Customer points":  round(customer_points, 1),
            "Max points":       round(max_points, 1),
            "Points gap":       round(max_points - customer_points, 1),
        })

    df_rc = (pd.DataFrame(rows)
               .sort_values("Points gap", ascending=False)
               .reset_index(drop=True))
    df_rc.insert(0, "Rank", range(1, len(df_rc)+1))
    return df_rc.head(top_n)

# ── Demo: lấy 3 khách hàng mẫu ───────────────────────────────────────────────
with open("model_lr.pkl", "rb") as f: lr_model_rc = pickle.load(f)

# Load scorecard
df_raw_clean = pd.read_csv("cs_training_cleaned.csv")

# Reload card (cần chạy lại sc.scorecard nếu chưa có trong memory)
# Nếu có lỗi ở đây, chạy lại Cell 7 của Bước 3 trước
try:
    card  # check if exists
    print("✓ Card loaded from memory")
except NameError:
    print("⚠️  Cần reload card — chạy lại Cell 7 của Bước 3 trước")

# Lấy 3 khách hàng bị reject (score thấp)
rejected = score_all[score_all["score"] < 550].head(3)
print(f"\nDemo reason codes cho 3 khách hàng bị reject (score < 550):")
print("=" * 60)
for i, (idx, row) in enumerate(rejected.iterrows()):
    customer = df_raw_clean.iloc[idx]
    print(f"\n--- Khách hàng #{i+1} | Score = {row['score']:.0f} | Target = {row['target']} ---")
    rc = get_reason_codes(customer, card, bins_final, top_n=5)
    print(rc[["Rank","Variable","Customer bin","Customer points","Max points","Points gap"]].to_string(index=False))


## 8. Validation Report — Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)

auc_train = roc_auc_score(y_train, y_prob_train)
gini_test = 2 * auc_test - 1
fpr_c, tpr_c, _ = roc_curve(y_test, y_prob_test)
ks_stat = (tpr_c - fpr_c).max()

# ── 1. Metric cards (top row) ─────────────────────────────────────────────────
metrics = [
    ("AUC",         f"{auc_test:.3f}",         "Test set",      auc_test > 0.75),
    ("Gini",        f"{gini_test*100:.1f}%",    "Test set",      gini_test > 0.5),
    ("KS",          f"{ks_stat*100:.1f}%",      "Test set",      ks_stat > 0.4),
    ("Brier Score", f"{brier_raw:.4f}",         "Lower = better",brier_raw < 0.06),
]
for i, (name, val, sub, is_good) in enumerate(metrics):
    ax = fig.add_subplot(gs[0, i])
    color = "#E8F5E9" if is_good else "#FFF8E1"
    ax.set_facecolor(color)
    ax.text(0.5, 0.65, val, ha="center", va="center",
            fontsize=26, fontweight="bold", transform=ax.transAxes,
            color="#1B5E20" if is_good else "#E65100")
    ax.text(0.5, 0.30, name, ha="center", va="center",
            fontsize=12, fontweight="bold", transform=ax.transAxes)
    ax.text(0.5, 0.12, sub, ha="center", va="center",
            fontsize=9, color="gray", transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

# ── 2. ROC Curve ──────────────────────────────────────────────────────────────
ax_roc = fig.add_subplot(gs[1, 0:2])
ax_roc.plot(fpr_c, tpr_c, color=PALETTE["good"], lw=2,
            label=f"ROC (AUC={auc_test:.3f}, Gini={gini_test*100:.1f}%)")
ax_roc.plot([0,1],[0,1], "k--", lw=1, alpha=0.5, label="Random")
ax_roc.fill_between(fpr_c, tpr_c, alpha=0.06, color=PALETTE["good"])
ax_roc.set_xlabel("FPR"); ax_roc.set_ylabel("TPR")
ax_roc.set_title("ROC Curve")
ax_roc.legend(fontsize=9, frameon=False)

# ── 3. Gains Chart ────────────────────────────────────────────────────────────
ax_gains = fig.add_subplot(gs[1, 2:4])
ax_gains.plot([0]+list(gl_table["Cumulative N %"]),
              [0]+list(gl_table["Cumulative Bad %"]),
              color=PALETTE["bad"], lw=2.5, marker="o", ms=5, label="Model")
ax_gains.plot([0,100],[0,100],"k--", lw=1, alpha=0.5, label="Random")
ax_gains.fill_between(
    [0]+list(gl_table["Cumulative N %"]),
    [0]+list(gl_table["Cumulative Bad %"]),
    [0]+list(gl_table["Cumulative N %"]),
    alpha=0.07, color=PALETTE["bad"]
)
ax_gains.set_xlabel("% Population"); ax_gains.set_ylabel("% Bad Captured")
ax_gains.set_title("Cumulative Gains")
ax_gains.legend(fontsize=9, frameon=False)

plt.suptitle("Credit Scorecard — Validation Report Dashboard",
             fontsize=15, fontweight="bold", y=1.01)
plt.savefig("18_validation_dashboard.png", bbox_inches="tight", dpi=130)
plt.show()
print("✓ Saved: 18_validation_dashboard.png")


In [ ]:
from datetime import date

auc_train_val = roc_auc_score(y_train, y_prob_train)
gini_val      = round((2 * auc_test - 1) * 100, 2)
ks_val        = round(ks_stat * 100, 2)

model_card_md = f"""# Model Card — Credit Risk Scorecard

**Version:** 1.0  
**Date:** {date.today().strftime("%Y-%m-%d")}  
**Author:** [Your Name]  
**Status:** Development / Portfolio Project

---

## 1. Model Overview

| Field | Detail |
|---|---|
| Model type | Logistic Regression with WOE encoding |
| Target variable | `target` — P(serious delinquency within 2 years) |
| Training dataset | Give Me Some Credit (Kaggle, 2011) |
| Training size | {len(X_train):,} records |
| Test size | {len(X_test):,} records |
| Score range | 300 – 850 (PDO=20, Base=600, Odds₀=1/19) |

---

## 2. Features Used ({len(WOE_COLS_FINAL)} variables)

| Variable | Description | IV Strength |
|---|---|---|
| `revolving_util` | Revolving credit utilization rate | Strong |
| `late_90_days` | Number of times 90+ days past due | Strong |
| `past_due_30_59` | Number of times 30-59 days past due | Strong |
| `past_due_60_89` | Number of times 60-89 days past due | Medium |
| `age` | Age of borrower | Medium |
| `debt_ratio` | Monthly debt payments / gross income | Medium |
| `monthly_income` | Monthly gross income | Medium |
| `real_estate_loans` | Number of real estate loans | Weak |
| `dependents` | Number of dependents | Weak |

**Excluded:** `open_credit_lines` (sign reversal — non-monotone relationship)

---

## 3. Performance Metrics

| Metric | Train | Test | Benchmark |
|---|---|---|---|
| AUC | {auc_train_val:.4f} | {auc_test:.4f} | > 0.75 ✓ |
| Gini | — | {gini_val:.2f}% | > 50% ✓ |
| KS | — | {ks_val:.2f}% | > 40% ✓ |
| Brier Score | — | {brier_raw:.4f} | < 0.06 ✓ |
| PSI (train vs test) | — | {psi_total:.4f} | < 0.10 ✓ |

---

## 4. Known Limitations & Risks

1. **Data vintage:** Dataset từ năm 2009 (US financial crisis) — có thể không đại diện cho điều kiện thị trường hiện tại
2. **Geography:** Dữ liệu từ thị trường Mỹ — cần validation trước khi áp dụng cho Việt Nam
3. **Missing income:** `monthly_income` missing 19.8% → imputed bằng median → có thể bias predictions cho nhóm này
4. **Open credit lines:** Bị loại do sign reversal — có thể capture thêm predictive power nếu dùng non-linear model
5. **Class imbalance:** Bad rate chỉ 6.7% — cần monitor nếu bad rate thực tế thay đổi đáng kể

---

## 5. Monitoring Plan

| Metric | Frequency | Action threshold | Action |
|---|---|---|---|
| PSI | Monthly | > 0.10 | Alert, investigate |
| PSI | Monthly | > 0.25 | Retrain model |
| Gini | Quarterly | Drop > 10 pts | Root cause analysis |
| Bad rate by band | Monthly | ± 20% vs baseline | Review cutoffs |
| Exceptions (Basel) | Monthly | Yellow zone | Increase monitoring |
| Exceptions (Basel) | Monthly | Red zone | Model review |

---

## 6. Ethical Considerations

- Model không sử dụng các biến protected attributes (race, gender, religion)
- Tuổi được dùng nhưng theo cách neutral (experience proxy, không phải ageism)
- Reason codes được implement để đảm bảo quyền giải thích của khách hàng (ECOA compliance)
- Khuyến nghị human review cho các quyết định borderline (score 560–620)

---

## 7. Files & Artifacts

| File | Description |
|---|---|
| `model_lr.pkl` | Trained Logistic Regression model |
| `woe_bins.pkl` | WOE binning definitions |
| `scorecard_points.csv` | Scorecard lookup table |
| `cs_scores.csv` | Score for all training records |
| `psi_report.csv` | PSI analysis report |
| `model_metadata.json` | Model parameters & metrics |

---

*This model card follows the Google Model Cards framework (Mitchell et al., 2019)*
"""

with open("model_card.md", "w", encoding="utf-8") as f:
    f.write(model_card_md)

print("✓ Saved: model_card.md")
print()
print(model_card_md[:800] + "...\n[truncated — xem file model_card.md để đọc đầy đủ]")


## 9. Xuất Model Card (Markdown)